# BIDS Conversion of EEG Data

**Author:** Noah Mba, noah.mba@fu-berlin.de  
**Date:** July 13, 2026  
**AI Acknowledgements:** Co-authored/Supported by Gemini and Claude 5 Sonnet  

---

This notebook performs the conversion of raw BrainVision EEG data to the BIDS standard for a batch of participants. The workflow involves reading the raw data, applying a standard 10-20 montage to define sensor positions, mapping experimental annotations to event triggers, and writing the dataset to a BIDS-compliant directory structure.

The BIDS writer automatically creates the correct folder structure in .../data/bids/ (i.e., sub-XX/eeg/), saves all EEG files (.vhdr, .vmrk, .eeg), generates event and channel metadata, CapTrak electrode and coordinate system files, and a sub-XX_scans.tsv file.

### 1. Setup: Loading modules, creating paths and directories

This initialization cell prepares the workspace by loading the relevant modules and specifying the relevant paths and directories.

In [6]:
import os
from pathlib import Path
import pandas as pd
import mne
from mne.channels import make_standard_montage
from mne_bids import BIDSPath, write_raw_bids

# Retrieves working directory of this notebook ('project/scripts/eeg')
notebook_dir = Path.cwd()

# Create path to main directory by going up two levels ('project')
project_root = notebook_dir.parent.parent

# Define paths to data folders
raw_dir = project_root / "data" / "raw"
bids_dir = project_root / "data" / "bids"
derivatives_dir = project_root / "data" / "derivatives"

# Set path to 'Quality Check' (QC) table in 'derivatives' folder
qc_file_path = derivatives_dir / "eeg_bids_conversion_qc_report.csv"

# Create 'derivatives' folder if it does not exist yet
derivatives_dir.mkdir(parents=True, exist_ok=True)

### 1. Creating the trigger - event mapping

The trigger map was specified before implementing the experiment in PsychoPy and is used to recode the numerical trigger codes from the EEG recording back to meaningful event codes.

In [7]:
trigger_map = {
    # --- EXPERIMENT META ---
    'PLACEHOLDER': 99999,
    'psychopy_enc_starts': 1,
    'system_255': 255,  # To resolve unexpected hardware or system triggers (appeared only in sub-27, unknown source)

    # ----------------------------------
    # --- PHASE 0: PRACTICE (2-3) ---
    # ----------------------------------
    'prac_start': 2, #should appear once before practice trials, e.g., at "initialize" routine
    'prac_end': 3,   #should appear once after practice trials, e.g., at "practice end" routine. 

    # ----------------------------------
    # --- PHASE 1: ENCODING (100-199) loop ---
    # ----------------------------------
    'enc_start': 100, 

    'enc_fixation': 110, 
    'enc_cue': 101,
    'enc_act1': 102,
    'enc_act2': 103,
    'enc_act3': 104, 

    # Targets (The 2x2 Manipulation) in "target_trial" routine
    'tgt_sc_pc': 151,
    'tgt_sc_pi': 152,
    'tgt_si_pc': 153,
    'tgt_si_pi': 154,

    # Encoding Questions in "response" routine
    'enc_q_schema': 161, #Did the object reach the intended recipient?
    'enc_q_color': 162,  #Was the last panel colorized?

    # Encoding Responses (Left/Right)
    'enc_resp_left': 171,
    'enc_resp_right': 172,

    # Encoding phase ends
    'enc_end': 199,

    # ----------------------------------
    # --- PHASE 2: DISTRACTOR (50-51) ---
    # ----------------------------------
    'dist_start': 50,
    'dist_end': 51,

    # ----------------------------------
    # --- PHASE 3: RETRIEVAL (200+) ---
    # ----------------------------------
    'psychopy_ret_starts': 4,
    'ret_start': 200, 

    'ret_fixation': 210, 

    # A. Cue Display (Either 1 of 4 OLD conditions or 1 NEW condition)
    'ret_cue_old_sc_pc': 211,
    'ret_cue_old_sc_pi': 212,
    'ret_cue_old_si_pc': 213,
    'ret_cue_old_si_pi': 214,
    'ret_cue_new': 215,

    # B. Question 1: "Seen before?" (Either 1 of 4 OLD conditions or 1 NEW condition) during "cue_recog" routine
    'ret_q1_old_sc_pc': 221,
    'ret_q1_old_sc_pi': 222,
    'ret_q1_old_si_pc': 223,
    'ret_q1_old_si_pi': 224,
    'ret_q1_new': 225,

    'ret_q1_resp_left': 228,  
    'ret_q1_resp_right': 229, 

    # C. Question 2: "Unexpected?" (Either 1 of 4 OLD conditions or 1 NEW condition) during "ending_recall" routine
    'ret_q2_old_sc_pc': 231,
    'ret_q2_old_sc_pi': 232,
    'ret_q2_old_si_pc': 233,
    'ret_q2_old_si_pi': 234,
    'ret_q2_new': 235,

    'ret_q2_resp_left': 238,  
    'ret_q2_resp_right': 239, 

    # D. Target Selection (4 Choices) (Either 1 of 4 OLD conditions or 1 NEW condition) during "target_trial" routine
    'ret_q3_old_sc_pc': 241,
    'ret_q3_old_sc_pi': 242,
    'ret_q3_old_si_pc': 243,
    'ret_q3_old_si_pi': 244,
    'ret_q3_new': 245,

    'ret_q3_resp_1': 251,
    'ret_q3_resp_2': 252,
    'ret_q3_resp_3': 253,
    'ret_q3_resp_4': 254,

    'ret_end': 209
}

### 3. BIDS Conversion
This cell performs the actual conversion of the raw EEG files into BIDS format. Importantly, the correct montage is loaded, defining the electrode layout for our experiment (both EEG and EOG electrodes). Also, the trigger map is saved as the event annotation in the newly created MNE raw object.

Additionally, it creates a quality check report that is saved as `eeg_bids_conversion_qc_report.csv` in the `derivatives` folder containing measurement metadata.

In [8]:
# 1. Create list of subjects to process (max. range: sub-03 bis sub-44)
subjects = [f"{i:02d}" for i in range(3, 45)]

# Initialize list for QC data
qc_data = []

for subj in subjects:
    print(f"\n--- Verarbeite sub-{subj} ---")
    
    sub_dir = Path(f"s{subj}") / "eeg"
    raw_fname = raw_dir / sub_dir / f"loc_s{subj}.vhdr"
    
    try:
        # CHANGED: preload=True — required so the correction below actually
        # modifies the in-memory data array, not just metadata
        raw = mne.io.read_raw(raw_fname, preload=True)
        
        raw.set_channel_types({'HEOG': 'eog', 'VEOG': 'eog'})

        # ==========================================
        # FIX: correct miscalibrated EOG channels
        # ==========================================
        # Bug: HEOG/VEOG have cal=1.0/range=1.0 in the BrainVision header
        # (vs. e.g. cal=0.0488281/range=1e-06 for EEG channels), so their
        # values are stored uncorrected in what appears to be µV, not volts.
        eog_cal_fixed = []
        for ch_name in ['HEOG', 'VEOG']:
            idx = raw.ch_names.index(ch_name)
            cal = raw.info['chs'][idx]['cal']
            rng = raw.info['chs'][idx]['range']
            # Guard: only correct if cal/range still show the uncorrected default.
            # Prevents double-applying the 1e-6 factor if this is ever rerun
            # on a file where it's already fixed upstream.
            if cal == 1.0 and rng == 1.0:
                raw.apply_function(lambda x: x * 1e-6, picks=[ch_name])
                eog_cal_fixed.append(ch_name)
        if eog_cal_fixed:
            print(f"  -> Corrected EOG calibration for: {eog_cal_fixed}")
        else:
            print(f"  -> INFO: EOG channels already calibrated, no correction applied")
        # ==========================================

        montage = make_standard_montage("standard_1020")
        raw.set_montage(montage, on_missing='warn')

        events, event_id = mne.events_from_annotations(raw, verbose='warning')
        
        info = raw.info
        nchan = info['nchan']
        ch_types = raw.get_channel_types()
        n_eeg = ch_types.count('eeg')
        n_eog = ch_types.count('eog')
        sfreq = info['sfreq']
        highpass = info['highpass']
        lowpass = info['lowpass']
        meas_date = info['meas_date']
        bads = info['bads']
        
        if nchan != 65:
            print(f"  -> ACHTUNG: sub-{subj} hat {nchan} Kanäle (Erwartet: 65)")
        if n_eeg != 63:
            print(f"  -> ACHTUNG: sub-{subj} hat {n_eeg} EEG-Kanäle (Erwartet: 63)")
        if n_eog != 2:
            print(f"  -> ACHTUNG: sub-{subj} hat {n_eog} Misc-Kanäle (Erwartet: 2)")
        if len(bads) > 0:
            print(f"  -> INFO: sub-{subj} hat markierte Bad Channels: {bads}")

        # ADDED: log EOG std post-fix so the QC table gives you a paper trail
        eog_data = raw.get_data(picks=['HEOG', 'VEOG'])
        eog_std = eog_data.std()
            
        qc_data.append({
            'Subject': f"sub-{subj}",
            'n_Channels': nchan,
            'n_EEG': n_eeg,
            'n_EOG': n_eog,
            'Sampling_Rate': sfreq,
            'Highpass_Hz': highpass,
            'Lowpass_Hz': lowpass,
            'Meas_Date': meas_date,
            'n_Bad_Channels': len(bads),
            'Bad_Channels': ", ".join(bads) if bads else "None",
            'EOG_cal_corrected': ", ".join(eog_cal_fixed) if eog_cal_fixed else "None",
            'EOG_std_postfix': round(eog_std, 8),
        })

        raw.set_annotations(None)

        bids_path = BIDSPath(
            subject=subj,
            datatype="eeg", 
            task="loc",
            root=bids_dir
        )
        bids_path.update(suffix="eeg", extension=".vhdr")

        write_raw_bids(
            raw,
            bids_path=bids_path,
            events=events,
            event_id=trigger_map,
            overwrite=True,
            allow_preload=True,
            format='BrainVision',   # keeps you in the same format as your source data
            verbose='warning',
        )

    except Exception as e:
        print(f"  -> ERROR for sub-{subj}: {e}")
        qc_data.append({'Subject': f"sub-{subj}", 'Bad_Channels': f"ERROR: {e}"})

df_qc = pd.DataFrame(qc_data)
df_qc.to_csv(qc_file_path, index=False)
print("\nAll subjects processed...")


--- Verarbeite sub-03 ---
Extracting parameters from c:\Users\noahm\projects\loc_analysis\data\raw\s03\eeg\loc_s03.vhdr...
Setting channel info structure...
Reading 0 ... 2982939  =      0.000 ...  2982.939 secs...


C:\Users\noahm\AppData\Local\Temp\ipykernel_22964\3254695503.py:18: RuntimeWarning: The unit for channel(s) HEOG, VEOG has changed from C to V.
  raw.set_channel_types({'HEOG': 'eog', 'VEOG': 'eog'})


  -> Corrected EOG calibration for: ['HEOG', 'VEOG']


C:\Users\noahm\AppData\Local\Temp\ipykernel_22964\3254695503.py:97: RuntimeWarning: Converting data files to BrainVision format
  write_raw_bids(



--- Verarbeite sub-04 ---
Extracting parameters from c:\Users\noahm\projects\loc_analysis\data\raw\s04\eeg\loc_s04.vhdr...
Setting channel info structure...
Reading 0 ... 2484799  =      0.000 ...  2484.799 secs...


...


--- Verarbeite sub-44 ---
Extracting parameters from c:\Users\noahm\projects\loc_analysis\data\raw\s44\eeg\loc_s44.vhdr...
Setting channel info structure...
Reading 0 ... 2350299  =      0.000 ...  2350.299 secs...


C:\Users\noahm\AppData\Local\Temp\ipykernel_22964\3254695503.py:18: RuntimeWarning: The unit for channel(s) HEOG, VEOG has changed from C to V.
  raw.set_channel_types({'HEOG': 'eog', 'VEOG': 'eog'})


  -> Corrected EOG calibration for: ['HEOG', 'VEOG']


C:\Users\noahm\AppData\Local\Temp\ipykernel_22964\3254695503.py:97: RuntimeWarning: Converting data files to BrainVision format
  write_raw_bids(



All subjects processed...
